# Learning Unit 7 - Evaluating models with XAI tools

In this assignment, we will explore how machine learning models make decisions. We will look into specialised tools designed to explain AI decisions, as well as models that offer interpretability from the outset — so-called 'white-box' models. We will continue to work with the CDC Diabetes Health Indicators dataset. To keep things simple, we will use the binary target version this time.

---
## Whitebox models

In this section, you will train a white-box model. White-box models are defined as being less complex and having humanly interpretable mathematics behind their predictions.

### Logistic Regression
One often used white-box model is the Logistic Regression. For logistic regression models, we can interpret the coefficients and constants (intercept) to understand the impact of each feature on the outcome (binary dependent variable). For example, with a logistic regression we can predict whether a patient has diabetes or not.

Here is a reminder about logistic regressions as described in https://pmc.ncbi.nlm.nih.gov/articles/PMC3936971/:

*"A logistic regression will model the chance of an outcome based on individual characteristics. Because chance is a ratio, what will actually be modeled is the logarithm of the chance given by:*

$$
\log\left(\frac{\pi}{1 - \pi}\right) = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_m x_m \tag{2}
$$

*where $ \pi $ indicates the probability of an event (e.g., death of a patient), and $ \beta_i $ are the regression coefficients associated with the reference group and the $ x_i $ explanatory variables. At this point, an important concept must be highlighted. The reference group, represented by $ \beta_0 $, is constituted by those individuals presenting the reference level of each and every variable $ x_1, x_2, \ldots $. To illustrate, considering our previous example, these are the individuals aged older that received standard treatment. Later, we will discuss how to set the reference level."*

**✏️ Task 7.1**  (Optional, code is provided)

1. *Train a logistic regression (see https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) on the CDC Diabetes Health Indicators dataset. This time, use the provided csv file with the binary indicators.*
2. *Print the model's coefficients*

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# Load the Diabetes data set
df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')

X = df.drop(columns=["Diabetes_binary"]).astype('int') # We only have binary and ordinal values
y = df.Diabetes_binary

# No scaling needed.

# Fit Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# Inspect coefficients & intercept
intercept = model.intercept_[0]
coefficients = model.coef_[0]

print(f"Intercept: {intercept:.3f} (odds ratio: {np.exp(intercept):.3f})\n")

for feature, coef in zip(X.columns, coefficients):
    print(f"{feature}: {coef:.3f} (odds ratio: {np.exp(coef):.3f})")


Intercept: -7.811 (odds ratio: 0.000)

HighBP: 0.756 (odds ratio: 2.131)
HighChol: 0.578 (odds ratio: 1.782)
CholCheck: 1.210 (odds ratio: 3.353)
BMI: 0.061 (odds ratio: 1.063)
Smoker: -0.009 (odds ratio: 0.991)
Stroke: 0.137 (odds ratio: 1.147)
HeartDiseaseorAttack: 0.221 (odds ratio: 1.247)
PhysActivity: -0.051 (odds ratio: 0.951)
Fruits: -0.050 (odds ratio: 0.951)
Veggies: -0.033 (odds ratio: 0.968)
HvyAlcoholConsump: -0.765 (odds ratio: 0.465)
AnyHealthcare: 0.079 (odds ratio: 1.082)
NoDocbcCost: 0.023 (odds ratio: 1.023)
GenHlth: 0.536 (odds ratio: 1.709)
MentHlth: -0.004 (odds ratio: 0.996)
PhysHlth: -0.007 (odds ratio: 0.993)
DiffWalk: 0.122 (odds ratio: 1.130)
Sex: 0.257 (odds ratio: 1.294)
Age: 0.124 (odds ratio: 1.132)
Education: -0.031 (odds ratio: 0.970)
Income: -0.051 (odds ratio: 0.950)


### Interpreting the model

<!-- Before the answer write "My answer: ", only for the next answer. -->
**✏️ Task 7.2**  
1. *Interpret the intercept of the logistic regression.*
2. *Interpret at least two coefficients of your model.*

**Helpful ressources:**
- The coefficient for the constant (______) indicates that when all predictor variables are zero, the log-odds of the outcome being 1 (1 meaning ______) is ______.
- Holding all other predictor variables at a fixed value, the odds of ______ (e.g., having diabetes) for patients with the binary characteristic  ______ (______ = 1) over the odds of ______ (e.g., having diabetes) for patients with the binary characteristic  ______ (______ = 0) is exp(______) = ______.  In terms of percent change, we can say that the odds for females are 166% higher than the odds for males.  
- Many online ressources (e.g., https://stats.oarc.ucla.edu/other/mult-pkg/faq/general/faq-how-do-i-interpret-odds-ratios-in-logistic-regression/, https://pmc.ncbi.nlm.nih.gov/articles/PMC3936971) show example interpretations. 


---
## Black-box models

Black-box models offer an alternative to white-box models. While black-box models often achieve higher predictive accuracy, they tend to be less interpretable than their white-box counterparts. 

### Multi Layer Perceptron

**✏️ Task 7.3**  (optional, code for MLP provided)
1. *As with the white-box models above, train a black-box model on the dataset. Specifically, use a RandomForestClassifier (see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html).* 
2. *Print the model's classification report*


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

# Load Data
df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')

X = df.drop(columns=["Diabetes_binary"]).astype('int') # We only have binary and ordinal values
y = df.Diabetes_binary

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Only binary and ordinal columns present, retrieve them
binary_columns = []
ordinal_columns = []

for col in X.columns:
    if len(np.unique(X[col])) == 2:
        binary_columns.append(col)
    else:
        ordinal_columns.append(col)

# Define appropriate scaling for MLP classifier
preprocessor = ColumnTransformer(
    transformers=[
        ("ord", StandardScaler(), ordinal_columns),
        ("bin", "passthrough", binary_columns)  # pass binary columns unchanged
    ],
    remainder='drop'
)

pipeline = make_pipeline(preprocessor, MLPClassifier(random_state=42, max_iter=500))

# Preprocess Train data
pipeline.fit(X_train, y_train)

# Train MLP classifier
model = MLPClassifier(random_state=42, max_iter=500)
model.fit(X_train, y_train)

# Predict on test data
y_pred = pipeline.predict(X_test)

# Print classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.88      0.97      0.93     54657
         1.0       0.53      0.21      0.30      8763

    accuracy                           0.87     63420
   macro avg       0.71      0.59      0.61     63420
weighted avg       0.84      0.87      0.84     63420



### Interpreting the outcome: Explainable AI Tools

Several XAI approaches have been developed to allow the continued use of black-block models while gaining some interpretability. In this assignment, we will use the SHAP (SHapley Additive exPlanations) explanation method. However, we will not go into the technical details of this method. If you are interested, here is a brief explanation: https://christophm.github.io/interpretable-ml-book/shap.html.

Use the shap library to calculate SHAP values for the MLP classifier.

You can find SHAP's documentation here: https://shap.readthedocs.io/en/latest/

**✏️ Task 7.4 - Local Explanations**  
*Use the shap library to generate visualizations for the predictions of two individual cases of the test sample. Interpret the cases in terms of at least two features.*

In [25]:
# Your code here.

Your interpretation here:


**✏️ Task 7.5 - Global Explanations**  
*Now generate global SHAP visualizations and interpret global feature importance in terms of SHAP for at least two features. If a simple approach takes to much time to converge, try a sampling approach.*

---
## 📝 Feedback

We are interested in your feedback in order to improve this course. We will read all of your feedback and evaluate it. What you share may have a direct impact on the rest of the course or future iterations of it.

Write down your feedback on the lecture, the exercises, or the assignments in the Markdown cell below. Furthermore, please note the approximate time it took you to complete the assignment. You may also write about your insights, what you found interesting, or questions that you have.